# 3. Blending models, applying free rules, and the final result

This notebook combines the models trained in notebook 2, adds the zero-risk rules found in
notebook 1, writes the submission file, and then — most importantly — reports what actually
happened on Kaggle's leaderboard, including the parts that didn't go as hoped.

## 3.1 Load the base model predictions and check them individually

In [1]:
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y = (train.Will_Buy_EV == "Yes").astype(int).values

d = np.load("demo_run_predictions.npz")
names = ["lgb_s42", "xgb_s42"]
oofs = [d[f"{n}_oof"] for n in names]
tes = [d[f"{n}_test"] for n in names]
for n, o in zip(names, oofs):
    print(f"{n:10s} OOF AUC {roc_auc_score(y, o):.5f}")

lgb_s42    OOF AUC 0.94552
xgb_s42    OOF AUC 0.94556


## 3.2 Blend by rank, with weights chosen to maximize cross-validated AUC

Two reasons to blend ranks rather than raw probabilities:
1. LightGBM and XGBoost scores aren't on exactly the same scale, so averaging raw probabilities
   can quietly let one model dominate. Ranks (`rankdata`, scaled to 0-1) put every model on the
   same footing.
2. AUC only cares about *order*, not the exact probability values — so working in rank space
   matches what's actually being optimized.

The blend weight between the two models is found by directly maximizing out-of-fold AUC —
not guessed, not just averaged 50/50.

In [1]:
R_oof = np.column_stack([rankdata(o) / len(o) for o in oofs])
R_te = np.column_stack([rankdata(t) / len(t) for t in tes])

def neg_auc(w):
    w = np.abs(w)
    return -roc_auc_score(y, R_oof @ (w / w.sum()))

res = minimize(neg_auc, np.ones(len(names)) / len(names), method="Nelder-Mead", options={"maxiter": 300})
w = np.abs(res.x) / np.abs(res.x).sum()
print("blend weights:", dict(zip(names, np.round(w, 3).tolist())))
print(f"blended OOF AUC: {-res.fun:.5f}  (equal-weight OOF AUC: {-neg_auc(np.ones(len(names))):.5f})")

blend_oof = R_oof @ w
blend_test = R_te @ w

blend weights: {'lgb_s42': 0.388, 'xgb_s42': 0.612}
blended OOF AUC: 0.94559  (equal-weight OOF AUC: 0.94558)


At demo scale the two models are similar enough that blending barely moves the number —
that's expected with only two base models at reduced training. The production blend used **four**
base models (LightGBM and XGBoost, each with two random seeds), which gives the blend more
genuinely different "opinions" to combine, and moved OOF AUC from ~0.9460 (best single model) to
**0.94608** (see README for the full run's numbers).

## 3.3 Apply the four verified, zero-risk rules from notebook 1

These are not predictions — they're facts about the data, checked against hundreds to tens of
thousands of rows each with **zero exceptions**. Snapping predictions to the extreme for rows that
match them costs nothing in risk and gains a small amount of free accuracy.

In [1]:
def apply_rules(pred, df):
    pred = pred.copy()
    inc = df.Annual_Income_USD.astype(float).values
    com = df.Daily_Commute_km.astype(float).values
    concern = df.Environmental_Concern_Level.astype(float).values
    subsidy = (df.Subsidy_Available == "Yes").values
    anxiety = df.Range_Anxiety_Level.values

    always_buys = inc >= 170537
    never_buys = ((inc >= 31004) & (inc <= 41970)) | (com >= 83) | \
                 ((inc == 30000) & (~subsidy) & ((concern == 1) | (anxiety == "High")))
    pred[always_buys] = np.maximum(pred[always_buys], 0.9999)
    pred[never_buys] = np.minimum(pred[never_buys], 0.0001)
    return pred

oof_with_rules = apply_rules(blend_oof, train)
print(f"blended OOF AUC before rules: {roc_auc_score(y, blend_oof):.5f}")
print(f"blended OOF AUC after rules:  {roc_auc_score(y, oof_with_rules):.5f}")
final_test_pred = apply_rules(blend_test, test)

blended OOF AUC before rules: 0.94559
blended OOF AUC after rules:  0.94560


The gain from the rules alone is small — a few thousand rows out of 668,665, and the model
was already getting most of them right thanks to the target encoding. But it is a *free* gain:
these rows are either always or never buyers, by construction, so there's no scenario where
snapping them to the extreme hurts.

## 3.4 Write and sanity-check the submission file

In [1]:
sub = pd.DataFrame({"id": test.id.values, "Will_Buy_EV": final_test_pred})
assert sub.id.is_unique
assert len(sub) == len(test)
assert sub.Will_Buy_EV.between(0, 1).all()
assert np.isfinite(sub.Will_Buy_EV).all()
sub.to_csv("submission_demo.csv", index=False)
print(f"wrote submission_demo.csv — {sub.shape[0]:,} rows, id range {sub.id.min()}-{sub.id.max()}")
print(sub.head())

wrote submission_demo.csv — 286,571 rows, id range 668665-955235
       id  Will_Buy_EV
0  668665     0.561480
1  668666     0.522897
2  668667     0.283411
3  668668     0.262447
4  668669     0.553044


## 4. The actual result, end to end

| Stage | AUC | Where |
|---|---|---|
| Raw 13 columns, no feature engineering | 0.9414 | holdout |
| + frequency of exact income/commute values | 0.9424 | holdout |
| + nested (leak-free) target encoding | 0.9447 | holdout |
| + tuned shallow trees | 0.9449 | holdout |
| **Final blend: 4 models, 10-fold leak-free CV** | **0.94608** | cross-validated (OOF) |
| Same model, submitted | **0.94613** | **public** leaderboard |
| Best submission after blending with a public community model | 0.94657 | public leaderboard (peak rank 232) |
| **Final result** | **0.94544** | **private / final** leaderboard, rank 346 |

A few honest observations, not just a highlight reel:

**The own-model number barely moved between CV and the public leaderboard** (0.94608 →
0.94613) — that's exactly what a leak-free, properly cross-validated pipeline should do, and it's
the strongest evidence the methodology in notebooks 1-2 is sound. If there were a leak, the public
score would have come in noticeably *lower* than the CV estimate, not matched it.

**Blending with a public community notebook raised the public score further** (up to 0.94657),
and I was transparent about this at the time — Kaggle's public leaderboard only scores 20% of the
test set, and near a deadline, combining a validated model with other competitors' published
output via rank-averaging is a normal, allowed part of how these competitions are played. It is
not, however, the same achievement as the modeling work in notebooks 1-2.

**The final private-leaderboard result (0.94544) came in below the best public score (0.94657),
closer to my own CV-only number.** This is the generalization gap in action: the public score is
exact for the 20% of rows it's computed on, but it's a noisy estimate of the other 80%, and
submissions heavily tuned toward maximizing the visible 20% (mine included, in its final form)
tend to give some of that back on the hidden 80%. My own cross-validated pipeline, which never
used any public-leaderboard feedback to make decisions, held up closer to its training estimate —
which is the actual lesson worth keeping from this competition.

## 5. What I'd do differently next time

- **Trust cross-validation over the public leaderboard, always.** The public score is one noisy
  sample; CV, done correctly, is the more honest signal — and this project is the proof.
- **Pick final submissions by role, not by who's highest on the public board.** One CV-anchored,
  "boring but honest" pick, and one public-informed pick — not two public-chasing picks.
- **Spend less time blending other people's output near the deadline, more time earlier on
  finding more findings like the income-spike one in notebook 1.** That's where the real, durable
  accuracy came from; blending mostly just moved rank by chasing leaderboard noise.